# __Phase 1: Data Exploration and Preparation__

In [ ]:
import pandas  as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import StrMethodFormatter, FuncFormatter

In [ ]:
# see all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)  

#### __Accessible data:__
From foodforecast we have four data scources as .parquet files:
1) __Sales__: Containig sales data from Qtr 2 2025
2) __Stores__: Containing Store Metadata (Id, zipcode, weekly revenue mean Qtr 1 2025)
3) __Holidays__: Containing a list of holidays dates in and in the corresponding region
3) __weather__: Containnig weather data form the corresponding regions and time span.   

#### __Objective:__
1) __The overall objective__ ist to predict the sold_quantity of items for the next day and enrich existing features with publicly available data to optimize the predictions. This is a __regression__ issue.
2) __for this notebook:__ Understanding, cleaning the four data sources to have one single dataframe ready for modeling.

# __Data exploration and data cleaning__

## __Data: Sales__ 


In [ ]:
# loading data
sales = pd.read_parquet('20260218_144523_sales_data.parquet', engine='pyarrow')
display(sales.head())


### Data Structure



This table contains bakery sales data per day from one client with several stores of the federal district of germany (NRW; northrhine westphalia) in the 2nd quarter of 2025 (1. Apr-30. Jun).

__Variables:__

> __date:__ the day of sales [datetime64[ns]]

> __category_name:__ name of the item category in german [str]

> __item_id__: the unique item id [int64]

> __sold_quantity__: the sales amount of the item per day ---> (this is the __target__ variable) [float64]

> __price__: the item's price [float64]

> __store_id__: unique id of one store [int64]


### Overview

In [ ]:

# shape
print('Shape sales:', sales.shape, end='\n\n')

# info
print(sales.info())

# decription numeric and categorical
display(sales.describe())
display(sales.describe(include='str'))





### Data Cleaning

In [ ]:
#checks 

# any duplicated rows?
print('Duplicates in sales:', sales.duplicated().sum())  # -> no duplicates

# any missing values?
missing = sales.isna().sum()
print('\nMissing values:')
print(missing)



In [ ]:
##!!
# --> Missing values only in price

# but price has also 0 and negative values - why? promotion? discount?
price_gt_0 = sales[sales.price > 0].shape[0]
price_lte_0 = sales[sales.price <= 0].shape[0]

# all rows 
all_price = sales.price.count()

# percent less than or equal to 0
perc_lte_0 = price_lte_0 / all_price * 100

# calculate proportion of price =< 0
print('Price > 0 \t ->', price_gt_0)
print('Price <= 0 \t ->', price_lte_0, f'({perc_lte_0:.1f} %)')
print(f'{perc_lte_0:.1f} % of price column are 0 or below (2759 (0.4 %) rows below 0)')


##!! price == NaN -> sold_quantity always 0 --> can we delete price NaN rows, since you cannnot sell anything w/out a price

# price is not our target, so i would leave the rows in dataframe and replace missing values with 0
sales['price'] = sales.price.fillna(0)
#display(sales.isna().sum())

# no more missing values
print('\nNo more missing values in sales:')
display(sales.isna().sum())

### Outliers

In [ ]:
# investigating target
print(' - TARGET description - ')
print(sales.sold_quantity.describe())

# -------- Outliers ----------

# boxplot and IQR

# IQR

# calculating quantiles
q1, q2, q3 = sales.sold_quantity.quantile([0.25, 0.5, 0.75])

# inter quartile range
IQR = q3 - q1

whiskers = 1.5

# outliers border
extreme_high = q3 + IQR * whiskers
extreme_low = q1 - IQR * whiskers

print('\nWhiskers factor:', whiskers)
print('\nValues high threshold:', extreme_high)
print('Values low threshold:', extreme_low)

# boxplot
plt.figure(figsize=(14,5))
sns.boxplot(x=sales.sold_quantity, whis=whiskers)
plt.grid(True, 'both', alpha=0.5)
plt.show()


# ----- questions and inconsistncies -------

# target values < 0 
sold_qty_gt_0 = sales[sales.sold_quantity > 0].shape[0]
sold_qty_lte_0 = sales[sales.sold_quantity <= 0].shape[0]

# all rows 
all_sold = sales.sold_quantity.count()

# percent less than or equal to 0
perc_lte_0 = sold_qty_lte_0 / all_sold * 100
 
print('-'*60)
print('\n- TARGET inconsistncies-')
print('sold_quantity  > 0 \t ->', sold_qty_gt_0)
print('sold_quantity <= 0 \t ->', sold_qty_lte_0, f'({perc_lte_0:.1f} %)')
print(f'\n{perc_lte_0:.1f} % of target column are 0 or below (2 rows only below 0 and can be neglected)')
print('How to deal with sold_quantity = 0?')

# ...
# count of outliers
outlier_cond = (sales.sold_quantity > extreme_high) | (sales.sold_quantity < extreme_low)
outliers = sales[outlier_cond].sort_values('sold_quantity', ascending=False)
outliers_count = outliers.shape[0]

# percent
perc_outliers = outliers_count / len(sales.sold_quantity) * 100

print('-'*60)
print(f'Count of outliers {outliers_count} ({perc_outliers:.2f} %).' )

# conclusion
print('Conclusion: Outliers seem to be significant and should not be erased.')

### Inconsistencies

In [ ]:
# strange
# there a items sold without a price but sold qty > 0

strange_solds_cond = (sales.sold_quantity != 0) & (sales.price == 0)

strange_solds = sales[strange_solds_cond] # -> more than 40,000 rows have sold items but no price. since our target it quantity sold, i would leave this untouched
display(strange_solds.sample(8))

strange_solds_count = strange_solds.shape[0]

perc_strange_solds = strange_solds_count / len(sales.sold_quantity) * 100

print(f'Count of items sold with price eq to 0: {strange_solds_count} ({perc_strange_solds:.2f} %).' )
print('\nHow do we deal with those occurences in price and sold_quantity?')
print('Assumption: These are results of promotion (buy one, get two, etc.) or give-a-ways (give away for free short before closing time) , so it \nseems reasonable that there are items giving away for free, i.e. without price.')

### Prepare for visualization

In [ ]:
# enrich dates for better analysis and visuaiisation
sales['month'] = sales.date.dt.month
sales['day_of_week'] = sales.date.dt.day_of_week
sales['week'] = sales.date.dt.isocalendar().week
# sales['day'] = sales.date.dt.day_name()  # -> not needed, too much RAM
sales.head()

## __Data: Stores__ 


In [ ]:
# loading stores

stores = pd.read_parquet('20260218_144523_stores.parquet', engine='pyarrow')
stores.head()


### Data Structure

This table contains store meta data of __serveral stores__ belonging to __one client__. 

__Variables:__

> __subdivision_code:__ The abbreviation for the federal state in Germany [str, categorical]

> __country_code__: The code of the country [str, categorical]

> __zipcode__: The postal code of the area where the store is located [int64, categorical]

> __average_weekly_revenue_Q1__: The average weekly revenue per store from Qtr 1 2025 (we have sales data form Qtr2) [float64, quantitative]

> __store_id__: The unique Id of a store [float64, categorical]

> 


### Overview

In [ ]:
# overview

# shape
print('Shape stores:', stores.shape, end='\n\n')

# info
print(stores.info())

# decription numeric and categorical
display(stores.describe())
display(stores.describe(include='str'))

### Data Cleaning

In [ ]:
## duplicates?
print('Dublicates in store:', stores.duplicated().sum())

# missing values
print('\nMissing values:')
print(stores.isna().sum())



In [ ]:
# deleting constant variables

print('Since the variables country_code and subdivision_code contain only ONE constant value, they do not provide any \
\nadditional information and can be deleted.')
stores = stores.drop(columns=['subdivision_code', 'country_code'], errors='ignore')

# check
display(stores.head())


In [ ]:
##!!
# -> investigating stores with average_weekly_revenue_Q1 == 0

# because of the description above the values of average_weekly_revenue_Q1 below the first (quartile, 25%) are 0.

#cond = stores.average_weekly_revenue_Q1 == 0
#stores[cond]

#xx = stores.groupby(['zipcode', 'store_id'])['average_weekly_revenue_Q1'].apply(lambda x: set(list(x)))
#pd.DataFrame(xx)

# sales with store_id > 60 
#sales_60 = (sales.store_id >= 59) #& (sales.sold_quantity > 0) #& 
#sales[sales_60].groupby(['store_id', 'category_name'])['item_id'].apply(lambda x: sorted(set(x)))

#display(sales[sales_60].describe())
#display(sales[sales_60].describe(include='str'))
#sales[sales_60].boxplot()

In [ ]:
# revenue qtr2 vs. qtr1


# creating a new column sales to calculate the revenue of Qtr 2
sales['revenue'] = sales.price * sales.sold_quantity
#display(sales.sort_values('revenue', ascending=False))

# creating revenue per store and per week for Qtr 2
rev_q2_week = pd.DataFrame(sales.groupby(['store_id', 'week'])['revenue'].sum())
rev_q2_week.sort_values('revenue', ascending=False)

# creating avgeage of weekly revenue Qtr 2
average_weekly_revenue_Q2 = rev_q2_week.groupby('store_id')['revenue'].mean()

# putting result in a dataframe and renaming the column
average_weekly_revenue_Q2 = pd.DataFrame(average_weekly_revenue_Q2).rename(columns={'revenue':'average_weekly_revenue_Q2'})

#display(average_weekly_revenue_Q2.sort_values('average_weekly_revenue_Q2', ascending=False).head(10))

#display(stores_revQ1.head(10))


# concat rev q2 to stores
# merge revenue vaiable for Qtr 2 and storr df (with revenue Qtr 1) into new df
stores_rev = pd.merge(left=stores, right=average_weekly_revenue_Q2, on='store_id')
stores_rev.sort_values('store_id').tail(30)


In [ ]:
# melting avg weekly columns for plotting
stores_rev = stores_rev.melt(id_vars='store_id', value_vars=['average_weekly_revenue_Q1', 'average_weekly_revenue_Q2'], var_name='Qtr', value_name='avg_weekly_rev')

# renaming values of Qtr to 1 or 2
stores_rev['Qtr'] = stores_rev.Qtr.str[24:]

stores_rev.head()

In [ ]:
# weekly rev qtr1 2025 by store: plot

plt.figure(figsize=(16,7))

#bars = plt.bar(stores.store_id, stores.avg_weekly_rev,  )
#bars = sns.barplot(data=stores, x='store_id', y='avg_weekly_rev', hue='Qtr', palette=['#F6B1CE', '#FF00FF'])
bars = sns.barplot(data=stores_rev, x='store_id', y='avg_weekly_rev', hue='Qtr', palette=['#F6B1CE', '#FF00FF'])
plt.grid(axis='y', alpha=0.6)
plt.title('Average Weekly Revenue Qtr 1 vs. Qtr 2')
plt.show()

In [ ]:
# conclusion store_id  >= 60
print('Stores with the Id of >= 60 have no revenue in Qtr 1 and Qtr 2, which means they did not sell anything. \nBut sometimes the sold_quantity column is > 0\
, sometime price is > 0.')

## __Data: Holidays__

In [ ]:
# loading data

holidays = pd.read_parquet('20260218_144523_holidays.parquet', engine='pyarrow')
holidays.head()


### Data Structure

This table contains a list of holiday of differnt typ.

__Variables:__

> __zipcode__: The postal code of the area where the holiday takes place. [str, categorical]

> __subdivision_code:__ The abbreviation for the federal state in Germany where the holiday takes place. [str, categorical]

> __date__: The date of the holiday of a holiday span. [datetime[ns], categorical]

> __holiday_name__: The name of the holiday(s) [str, categorical]

> __holiday_type__: The type of the holiday(s) [str, categorical]

### Overview


In [ ]:
# shape
print('Shape holidays:', holidays.shape, end='\n\n')

# info
print(holidays.info())

# decription numeric and categorical
display(holidays.describe())
display(holidays.describe(include='str'))

### Data Cleaning

In [ ]:
# duplicates?
print('Duplicates:', holidays.duplicated().sum())

# missing values?
print('Missing values:', holidays.isna().sum().sum())

# 
print('\nActually, a 2nd federal subdivision sneaked its way in...  -> ', list(holidays.subdivision_code.unique()))

# record with "wrong" subdivision_code
cond = holidays.subdivision_code == 'DE-BB'

# list with zipcodes in 2nd subdivision
subs = list(holidays[cond].zipcode.unique())

print('\nZipcode for different subdivision:', subs)
print(f'Conclusion: Since the subdivision_code {subs} belongs to "DE-NW", there seems to be a mistake. We would replace it with the correct code.')
print('BUT: Since the variable subdivision_code has only one constant value, it provides no additional information and should be deleted.')


holidays = holidays.drop(columns=['subdivision_code'])
holidays.head()

In [ ]:
##!! work in progress

# ostersonntag nad pfingstsonntag are only once counted.
print(holidays.holiday_name.value_counts())

cond = holidays.date == '2025-04-20'
holidays[cond]

xx = holidays.groupby(['zipcode', 'date'])['holiday_name'].apply(lambda x: set(x))

xx

### Outliers / Inconsistencies / Dependencies / Issues

### Prepare for visualization

In [ ]:
# adding a is_holiday boolean variable

holidays['is_holiday'] = 1

In [ ]:
# plotting sold_quantity per date and is_holiday


# adding date fragments
holidays['day_of_month'] = holidays.date.dt.day
holidays['week'] = holidays.date.dt.isocalendar().week
holidays

# plot
fig, ax = plt.subplots(1,1, figsize=(15,6))

sns.lineplot(sales, x='date', y='sold_quantity', label='sold_quantity', ax=ax, legend=False)

# 2nd y-axis on the right
ax2 = ax.twinx()
sns.scatterplot(holidays, x='date', y='is_holiday', color='r',label='holiday', ax=ax2, legend=False)

# getting labels of both axes
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
plt.legend(lines + lines2, labels + labels2, loc='upper left')

# all dates for grid
all_dates = pd.date_range(start=min(holidays['date'].min(), sales['date'].min()),
                          end=max(holidays['date'].max(), sales['date'].max()),
                          freq='D')

ax.set_xticks(all_dates[::15])  
ax.grid(True, which='both', axis='x', alpha=0.6) 


# additional lines per day
for d in all_dates:
    ax.axvline(d, color='gray', linestyle='-', alpha=0.2, zorder=0)

#ax.grid(True, alpha=0.6)
plt.title('Sales per date per holidays')
plt.xticks(rotation=65)
ax2.set_yticks([])
ax2.set_ylabel('')
plt.show()

## __Data: Weather__

In [ ]:
# data loading
weather = pd.read_parquet('20260218_144523_weather.parquet', engine='pyarrow')
display(weather.head())

### Overview

In [ ]:
# shape
print('Shape holidays:', weather.shape, end='\n\n')

# info
print(weather.info())

# decription numeric and categorical
display(weather.describe())
display(weather.describe(include='str'))

### Data Structure

> ...coming soon...

### Data Cleaning

In [ ]:
# duplicates?
print('Duplicates:', weather.duplicated().sum())

# missing values?
print('Missing values:', weather.isna().sum().sum())


In [ ]:
# first view issues

# investigating "chancesof..." columns, which seem to be always 0.

# boxplot
plt.figure(figsize=(15,5))
weather.boxplot()
plt.xticks(rotation=70)
plt.show()

In [ ]:
# getting all "chances of..." columns
chancesof_cols = list(weather.columns[19:-1])

weather_chancesof = weather[chancesof_cols]

display(weather_chancesof.value_counts())
print('-> Only 0 values in chancesof cols.')
print('-'*30)
print('Conclusion: Those columns only have one constant value (0), i.e. they do not provide information for a model -> we delete them.')

weather_reduced = weather.drop(columns=chancesof_cols)
weather_reduced.head()

### Inconsistencies

#### Functional dependencies / redundant information

In [ ]:
# weather_code and weather_description 
# seem to have redundant information

weather_code_decr = weather_reduced.groupby(['weather_code'])['weather_description'].apply(lambda x: list(set(x)))
print('Represantation of weather_code vs weather_decription - redundant information:')
print(weather_code_decr)
print('Conclusion: wether_description can be deleted.')



In [ ]:
# temperature and headindex 
# seem to have quite the information.

print('Temperature and Heatindex seem to be almost equal variables:')

# compare description and boxplot
print('\nDescription & Boxplot:')
temp_heat = weather_reduced[['temperature', 'heatindex']]

print(temp_heat.describe())

plt.figure()
temp_heat.boxplot()
plt.show()

In [ ]:
# comparing temperature vs. heatindex
#print(temp_heat.value_counts())
temp_vs_heat = weather_reduced.groupby('temperature')['heatindex'].apply(lambda x: list(set(x)))
temp_vs_heat_mean = weather_reduced.groupby('temperature')['heatindex'].apply(lambda x: int(np.mean(list(set(x)))))

samples = temp_vs_heat.sample(30)
print('1) Variables temperature vs. heatindex:\n')
print(samples)

print('-'*30)

samples_rounded = temp_vs_heat_mean
print('\n2) Variables temperature vs. heatindex [ROUNDED MEAN] :\n')
print(samples_rounded)

print('\nAnalysis: As we can see, the heatindex seems to measure more precisely (1) but the mean of the heatindex values is almost identical to temperature (2)\
      .So we can conclude those variables provide the same information and one of them can be deleted.' )

## --> which one to delete? temperature or heatindex?



In [ ]:
# windchill and feelslike 
# seem to have redundant information, too

print('Windchell and Fellslike seem to be almost equal variables:\n')
print('Description & Boxplot:')
wind_feel = weather_reduced[['windchill', 'feelslike']]

print(wind_feel.describe())

plt.figure()
wind_feel.boxplot()
plt.show()

In [ ]:
wind_chill = weather_reduced.groupby('windchill')['feelslike'].apply(lambda x: set(x))
wind_chill_mean = weather_reduced.groupby('windchill')['feelslike'].apply(lambda x: round(np.mean(list(set(x)))))

print('Comparison windchil vs feelslike [ROUNDED MEAN]')
print(wind_chill_mean[:10])
print('\nAnalysis: We can see the same behaviour as in the former realtionships, so one (windchill or feelslike) can be deleted due to redundance.')

# linechart for visual comparison
xxx = range(len(wind_chill))
plt.plot(xxx, wind_chill_mean, label='windchill')
plt.plot(xxx, wind_chill_mean.index, c='m', label='feelslike')
plt.grid(True)
plt.title('windchill vs. feelslike')
plt.legend()
plt.show()

### Prepare for visualisation

In [ ]:
# deleting redundant columns (choice: Marcel -> können wir auch ändern)

delete_cols = ['windchill', 'heatindex', 'weather_code']

# new df
weather_reduced_2 = weather_reduced.drop(columns=delete_cols)

weather_reduced_2.head()


In [ ]:
# reducing weather df to day level, eleminating the hourly records

# what to groupby and what to aggragate (mean, etc..)
weather_per_day = weather_reduced_2.groupby(['date','zipcode']).agg({
 'temperature': 'mean',
 'wind_speed': 'mean',
 'wind_degree': 'mean',
 'wind_dir': lambda x: x.mode()[0],
 'weather_description': lambda y: y.mode()[0], # make sense?
 'precip': 'mean',
 'humidity': 'mean',
 'visibility': 'mean',
 'pressure': 'mean',
 'cloudcover': 'mean',
 'dewpoint': 'mean',
 'windgust': 'mean',
 'feelslike': 'mean',
 'uv_index': 'mean'
 }
).reset_index()


# new weather df
display(weather_per_day.head())
print('Shape:', weather_per_day.shape)


## __Creating one final dataframe for modeling__

> ...coming soon...

# __Visualization__


#### __Sales per Week / Weekday__

In [ ]:
### check item_id in category per store
### ------------------- exkurs category_name -> item_id
### -----------
### -----------
### -----------

c = sales.category_name == 'Brotwaage'
sales[c]

pd.set_option("display.max_colwidth", None)

category_list = sales.category_name.unique()
for i, v in enumerate(category_list):
    print(f'{i} {v}')

sales_grp = pd.DataFrame(sales.groupby(['store_id', 'category_name'])['item_id'].apply(lambda x: sorted(x.unique())))
cond = sales_grp.index.get_level_values('category_name') == category_list[11]

sales_grp[cond]


In [ ]:
# sales per week and weekday

sales_per_week = sales.groupby('week')['sold_quantity'].sum()[:-1] # last week incomplete
#print('Sales per week:')
#print(sales_per_week)

qty_mean = sales_per_week.mean()
#print('Mean sold qty')
#print(qty_mean)

## -- plots

fig, ax = plt.subplots(1,2, figsize=(15,5))

ax[0].bar(sales_per_week.index, sales_per_week, color='m')
ax[0].set_xticks(sales_per_week.index)
ax[0].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x/1000)}K'))
ax[0].axhline(qty_mean, color='g', alpha=0.6, lw=2, ls='--', label=f'Average: {qty_mean/1e3:,.1f}K')

ax[0].set_xlabel('Week')
#ax[0].set_ylabel('QTY')

ax[0].set_title('Overall Sales per Week')


# ---------------------

sales_per_weekday = sales.groupby('day_of_week')['sold_quantity'].sum()
#print('Sales per day_of_week:')
#print(sales_per_weekday)

qty_mean = sales_per_weekday.mean()
#print('Mean sold qty')
#print(qty_mean)

bars = ax[1].bar(sales_per_weekday.index, sales_per_weekday, color='m')

ax[1].set_xticks(sales_per_weekday.index, ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax[1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x/1e6}M'))

ax[1].axhline(qty_mean, color='g', alpha=0.5, lw=2, ls='--', label=f'Average: {qty_mean/1e6:,.0f}M')

ax[1].set_xlabel('Day of Week', fontsize=8)
#ax[1].set_ylabel('QTY', fontsize=8)
ax[1].set_yticks([])

ax[1].set_title('Overall Sales per Weekday')

# adding bar labels on top
bar_labels = [f'{v/1e6:,.1f}M' for v in sales_per_weekday]
ax[1].bar_label(bars, labels=bar_labels , padding=3)


# ------ for all

for a in ax:
    a.grid(axis='y', alpha=.6)
    a.spines[['top', 'left', 'right']].set_visible(False)
    a.legend(loc='center', framealpha=0.95)

fig.suptitle('Overall Sales per Time  - (Qtr 2, 2025)', fontsize=16)
plt.show()

#### __Sales per Category / Item__

In [ ]:
# Top 10 Category ---

cat_sales = sales.groupby('category_name')['sold_quantity'].sum().sort_values(ascending=False)

#print('Overall sold_quantity by category:')
#display(cat_sales)

top_10_cat = cat_sales.head(10)

top_10_cat = pd.DataFrame(top_10_cat).sort_values('sold_quantity', ascending=True)

#display(top_10_cat)

## ----------------------------------------------

# Top 10 Sold Items ---

item_sales = sales.groupby('item_id')['sold_quantity'].sum().sort_values(ascending=False)

top_10_item = item_sales.sort_values(ascending=False).head(10)

top_10_item = pd.DataFrame(top_10_item).sort_values('sold_quantity')



In [ ]:
# Top 10 Category: plot


fig, ax = plt.subplots(1,2, figsize=(15,6))


# top 10 category
bar_cat = ax[0].barh( top_10_cat.index.astype(str), top_10_cat.sold_quantity, color='m')

bar_labels = [f'{v/1e3:.0f}K' if v < 1e6 else f'{v/1e6:.2f}M' for v in top_10_cat.sold_quantity]

ax[0].bar_label(bar_cat, labels=bar_labels, padding=2)

ax[0].set_ylabel('Product Category', fontsize=12)
ax[0].set_title('Product Category - Top 10')


# ----------------

# Top 10 Sold Items: plot


# top 10 category items
bars_item = ax[1].barh(top_10_item.index.astype(str), top_10_item.sold_quantity,  color='m')

bar_labels = [f'{v/1e3:.0f}K' if v < 1e6 else f'{v/1e6:.2f}M' for v in top_10_item.sold_quantity]

ax[1].bar_label(bars_item, labels=bar_labels, padding=2)

ax[1].set_ylabel('Category Item Id.', fontsize=12)
ax[1].set_title('Sold Items - Top 10')

# for all plots-----

for a in ax:
   #a.grid(axis='x', alpha=0.6)
    a.spines[['top', 'bottom', 'right']].set_visible(False)
    a.set_xticks([])

fig.suptitle('Overall Sales per Category and Category Item (Qtr 2, 2025)', fontsize=18)

plt.show()


In [ ]:
# Top 10 stores

top_10_stores = sales.groupby('store_id').agg(sold_qty_sum=('sold_quantity', 'sum'), sold_qty_std=('sold_quantity', 'std')).sort_values('sold_qty_sum', ascending=False)
top_10_stores = top_10_stores.head(10).sort_values('sold_qty_sum', ascending=True)
top_10_stores


In [ ]:
# op 10 stores: plot

fig, ax = plt.subplots(1,1, figsize=(8,6))

# top 10 category items
bars = ax.barh(top_10_stores.index.astype(str), top_10_stores.sold_qty_sum,  color='m')
bar_labels = [f'{v/1e3:.0f}K' if v < 1e6 else f'{v/1e6:.2f}M' for v in top_10_stores.sold_qty_sum]
ax.bar_label(bars, labels=bar_labels, padding=2)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
#ax.set_xlim(1.3e5, 3.6e5)

ax.set_xlabel('Sold Unites')
ax.set_ylabel('Store Id.', fontsize=8)
ax.set_title('Stores - Top 10 (Qtr2 2025)')

ax.grid(axis='x', alpha=0.6, )

plt.show()